In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import sys

project_root = '/content/drive/MyDrive/GNNs/HIV inhibitors-GNN'
os.makedirs(project_root, exist_ok=True)
os.chdir(project_root)
sys.path.append(project_root)

In [ ]:
import torch
torch_version = str(torch.__version__)
print("PyTorch has version {}".format(torch_version))
!pip install rdkit

scatter_src = f"https://pytorch-geometric.com/whl/torch-{torch_version}.html"
sparse_src = f"https://pytorch-geometric.com/whl/torch-{torch_version}.html"
!pip install torch-scatter -f $scatter_src
!pip install torch-sparse -f $sparse_src
!pip install torch-geometric


PyTorch has version 2.10.0+cu128
Looking in links: https://pytorch-geometric.com/whl/torch-2.10.0+cu128.html
  Using cached torch_scatter-2.1.2.tar.gz (108 kB)
  Preparing metadata (setup.py) ... done
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 423, in run
    _, build_failures = build(
                        ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/wheel_builder.py", line 319, in build
    wheel_file = _build_one(
                 ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/wheel_builder.py", line 193, 

In [ ]:

!pip install -q mlflow pyngrok

import os
from getpass import getpass
import mlflow
from pyngrok import ngrok
import time
import subprocess

BASE_DIR = "/content/drive/MyDrive/GNNs/HIV inhibitors-GNN"

os.makedirs(f"{BASE_DIR}/outputs/mlruns", exist_ok=True)
os.makedirs(f"{BASE_DIR}/outputs/artifacts", exist_ok=True)

MLFLOW_TRACKING_URI = f"sqlite:///{BASE_DIR}/outputs/mlruns/mlflow.db"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

EXPERIMENT_NAME = "HIV_GNN_Classification"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")

print("\nngrok authtoken............")
ngrok_token = getpass("Paste your ngrok authtoken here and press Enter: ")

if ngrok_token.strip():
    !ngrok config add-authtoken {ngrok_token}
    print("ngrok authtoken saved!")
else:
    print("No token entered. Public UI will not work.")

# === Kill old processes (clean start) ===
ngrok.kill()
!pkill -f "mlflow ui" || true
time.sleep(2)

print("\nStarting MLflow UI...")

# ✅ FIXED + IMPROVED command
subprocess.Popen([
    "mlflow", "ui",
    "--port", "5000",
    "--host", "0.0.0.0",                    # ← THIS IS THE KEY FIX
    "--backend-store-uri", MLFLOW_TRACKING_URI,
    "--default-artifact-root", f"file://{BASE_DIR}/outputs/artifacts"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait until the port is actually listening
print("Waiting for MLflow to bind to port 5000...")
for i in range(15):
    time.sleep(1)
    import socket
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    result = sock.connect_ex(('127.0.0.1', 5000))
    sock.close()
    if result == 0:
        print(f"✅ MLflow UI is ready on port 5000 (took {i+1}s)")
        break
    if i == 14:
        print("⚠️ MLflow did not start in time. Check the next lines.")

time.sleep(2)

# Start ngrok
try:
    public_url = ngrok.connect(5000, "http").public_url
    print(f"\n🎉 MLflow UI is live at: {public_url}")
    print("   → Open this link in a new tab")
    print("   → Keep this Colab cell running (do not close the tab)")
except Exception as e:
    print(f"\n❌ Could not create ngrok tunnel: {e}")

# === Diagnostic (very useful) ===
print("\n=== Running processes ===")
!ps aux | grep mlflow
print("\n=== Port 5000 status ===")
!lsof -i :5000

In [ ]:
%run src/training/train.py